# Orders

In [ ]:
# reading data from orders


conn = duckdb.connect('../database/olist.db')
# Execute and fetch as list of tuples

# Convert the base columns to real timestamps one by one
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE orders ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

result = conn.execute("select * from orders").df().head()
print(result)  # [(2,)]


In [ ]:
# calculating number of days till the approval, shipment, delivery

query = """ 
CREATE OR REPLACE TABLE orders_delivery_details AS 
SELECT 
    order_id, customer_id, order_status,
    datediff('day', order_purchase_timestamp, order_approved_at) as days_till_approval,
    datediff('day', order_purchase_timestamp, order_delivered_carrier_date ) as days_till_shipped,
    datediff('day', order_purchase_timestamp , order_delivered_customer_date ) as days_till_delivery,
    datediff('day', order_purchase_timestamp , order_estimated_delivery_date ) as days_till_estim_delivery,
    datediff('day', order_estimated_delivery_date, order_delivered_customer_date) AS delivery_accuracy
from orders """
conn.execute(query)


In [ ]:
# reading the order delivery details
query = """
    SELECT * from orders_delivery_details
"""
df=conn.execute(query).df()
print(df)